# Variant Analysis
Per-run variant deep-dives, cross-sample comparisons, overlap heatmaps, and same-donor similarity.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output
import scipy.stats as stats

DB_PATH = 'results/variants.db'

def query(sql, params=()):
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)

all_runs = query("""
    SELECT r.run_id, s.name AS sample, s.cell_line,
           r.mapping_rate, r.total_raw, r.total_filt
    FROM runs r
    JOIN samples s ON r.sample_id = s.sample_id
    ORDER BY r.run_id
""")
all_runs['label'] = all_runs['cell_line'].fillna(all_runs['sample'])
all_runs['data_type'] = all_runs['run_id'].apply(
    lambda x: 'WES' if '_wes_' in x.lower() else 'sc/bulk'
)

sc_runs  = all_runs[all_runs['data_type'] == 'sc/bulk'].copy()
wes_runs = all_runs[all_runs['data_type'] == 'WES'].copy()

print(f'Loaded {len(all_runs)} runs total — sc/bulk: {len(sc_runs)}, WES: {len(wes_runs)}')

## Per-run deep-dive
Select a run to see depth distribution, quality score distribution, allele frequency spectrum, variant types, and chromosomal distribution.

In [ ]:
run_out = widgets.Output()
run_select = widgets.Dropdown(description='Run:', layout=widgets.Layout(width='65%'))

def refresh_run_dropdown():
    run_labels = [f"{r['run_id']} ({r['label']})" for _, r in all_runs.iterrows()]
    run_select.options = list(zip(run_labels, all_runs['run_id']))

def show_run_detail(run_id):
    geno = query("""
        SELECT v.chromosome AS chrom, v.position AS pos, v.ref_allele AS ref,
               v.alt_allele AS alt, g.depth, g.quality AS qual, g.allele_freq,
               g.filter_status, v.variant_type
        FROM genotype_calls g
        JOIN runs r ON g.run_id = r.run_id
        JOIN variants v ON g.variant_id = v.variant_id
        WHERE r.run_id = ? AND g.filter_status = 'PASS'
        LIMIT 500000
    """, params=(run_id,))
    if geno.empty:
        print(f'No passing variants for {run_id}.')
        return

    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=(
            'Depth distribution', 'QUAL score distribution', 'Allele frequency spectrum',
            'Variant types', 'Chromosomal distribution', 'Depth vs AF'
        ),
        horizontal_spacing=0.1, vertical_spacing=0.15
    )
    fig.add_trace(go.Histogram(x=geno['depth'], nbinsx=60, marker_color='#457b9d'), row=1, col=1)
    fig.add_trace(go.Histogram(x=geno['qual'],  nbinsx=60, marker_color='#2d6a4f'), row=1, col=2)
    fig.add_trace(go.Histogram(x=geno['allele_freq'], nbinsx=50, marker_color='#e9c46a'), row=1, col=3)

    if geno['variant_type'].notna().any():
        vt = geno['variant_type'].value_counts()
        fig.add_trace(go.Bar(x=vt.index, y=vt.values, marker_color='#e76f51'), row=2, col=1)

    chrom_order = [str(c) for c in range(1, 23)] + ['X', 'Y', 'MT']
    chrom_counts = geno['chrom'].str.replace('chr', '', regex=False).value_counts().reindex(chrom_order).dropna()
    fig.add_trace(go.Bar(x=chrom_counts.index, y=chrom_counts.values, marker_color='#264653'), row=2, col=2)

    s = geno.sample(min(5000, len(geno)))
    fig.add_trace(go.Scatter(x=s['depth'], y=s['allele_freq'],
                             mode='markers', marker=dict(size=3, color='#a8dadc', opacity=0.5)), row=2, col=3)

    label = all_runs[all_runs['run_id'] == run_id]['label'].values
    title_label = label[0] if len(label) else run_id
    data_type = all_runs[all_runs['run_id'] == run_id]['data_type'].values
    dtype_str = data_type[0] if len(data_type) else ''
    fig.update_layout(height=600,
                      title=f"{run_id}  ({title_label})  —  {len(geno):,} passing variants  [{dtype_str}]",
                      showlegend=False)
    fig.show()

def on_run_change(change):
    with run_out:
        clear_output(wait=True)
        show_run_detail(change['new'])

run_select.observe(on_run_change, names='value')

refresh_run_dropdown()
display(run_select, run_out)
if run_select.options:
    show_run_detail(run_select.options[0][1])

## Cross-sample comparison
Shared variants, depth, and AF comparison between two selected runs.

In [ ]:
select_a = widgets.Dropdown(description='Run A:', layout=widgets.Layout(width='55%'))
select_b = widgets.Dropdown(description='Run B:', layout=widgets.Layout(width='55%'))
out_compare = widgets.Output()

def refresh_compare_dropdowns():
    run_labels = [f"{r['run_id']} ({r['label']})" for _, r in all_runs.iterrows()]
    opts = list(zip(run_labels, all_runs['run_id']))
    select_a.options = opts
    select_b.options = opts
    if len(opts) > 1:
        select_b.value = opts[1][1]

def compare_runs(run_a, run_b):
    if run_a == run_b:
        print('Select two different runs.')
        return
    geno_a = query("""
        SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS key,
               g.depth, g.allele_freq
        FROM genotype_calls g
        JOIN variants v ON g.variant_id = v.variant_id
        WHERE g.run_id=? AND g.filter_status='PASS'
    """, (run_a,))
    geno_b = query("""
        SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS key,
               g.depth, g.allele_freq
        FROM genotype_calls g
        JOIN variants v ON g.variant_id = v.variant_id
        WHERE g.run_id=? AND g.filter_status='PASS'
    """, (run_b,))
    if geno_a.empty or geno_b.empty:
        print('No passing variants for one or both runs.')
        return

    set_a, set_b = set(geno_a['key']), set(geno_b['key'])
    overlap = set_a & set_b
    jaccard = len(overlap) / len(set_a | set_b) if (set_a | set_b) else 0
    print(f'Variants — A: {len(set_a):,}  |  B: {len(set_b):,}  |  shared: {len(overlap):,}')
    print(f'Jaccard: {jaccard:.4f}  |  A⊂B: {len(overlap)/len(set_a):.4f}  |  B⊂A: {len(overlap)/len(set_b):.4f}')

    label_a = all_runs[all_runs['run_id'] == run_a]['label'].values[0]
    label_b = all_runs[all_runs['run_id'] == run_b]['label'].values[0]

    shared_keys = list(overlap)
    df_a = geno_a[geno_a['key'].isin(shared_keys)].set_index('key').add_suffix('_a')
    df_b = geno_b[geno_b['key'].isin(shared_keys)].set_index('key').add_suffix('_b')
    merged = df_a.join(df_b).dropna()

    fig = make_subplots(rows=1, cols=2, subplot_titles=('Depth at shared sites', 'AF at shared sites'))
    if not merged.empty:
        s = merged.sample(min(5000, len(merged)))
        fig.add_trace(go.Scatter(x=s['depth_a'], y=s['depth_b'],
                                 mode='markers', marker=dict(size=3, color='#264653', opacity=0.4)), row=1, col=1)
        fig.add_trace(go.Scatter(x=s['allele_freq_a'], y=s['allele_freq_b'],
                                 mode='markers', marker=dict(size=3, color='#e76f51', opacity=0.4)), row=1, col=2)
    for col in [1, 2]:
        fig.update_xaxes(title_text=label_a, row=1, col=col)
        fig.update_yaxes(title_text=label_b, row=1, col=col)
    fig.update_layout(height=420, title=f'Shared sites: {label_a} vs {label_b}', showlegend=False)
    fig.show()

def on_compare_change(_):
    with out_compare:
        clear_output(wait=True)
        compare_runs(select_a.value, select_b.value)

select_a.observe(on_compare_change, names='value')
select_b.observe(on_compare_change, names='value')

refresh_compare_dropdowns()
display(select_a, select_b, out_compare)
if select_a.options and select_b.options:
    compare_runs(select_a.value, select_b.value)

## Variant overlap heatmap
All-vs-all Jaccard similarity and containment matrix for all runs.

In [ ]:
def build_overlap_heatmap(runs_df, label):
    run_ids = runs_df['run_id'].tolist()
    labels  = runs_df['label'].tolist()
    n = len(run_ids)
    if n == 0:
        print(f'No {label} runs.')
        return None, None, None, None, None

    print(f'Loading variant keys for {n} {label} runs (may take a moment)...')
    all_keys = {}
    for run_id in run_ids:
        df = query("""
            SELECT v.chromosome||':'||v.position||':'||v.ref_allele||':'||v.alt_allele AS key
            FROM genotype_calls g
            JOIN variants v ON g.variant_id = v.variant_id
            WHERE g.run_id=? AND g.filter_status='PASS'
        """, (run_id,))
        all_keys[run_id] = set(df['key'].tolist()) if not df.empty else set()

    jaccard_mat = np.zeros((n, n))
    contain_mat = np.zeros((n, n))
    for i, ri in enumerate(run_ids):
        for j, rj in enumerate(run_ids):
            si, sj = all_keys[ri], all_keys[rj]
            union = len(si | sj)
            inter = len(si & sj)
            jaccard_mat[i, j] = inter / union if union else 0
            contain_mat[i, j] = inter / len(si) if si else 0

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Jaccard similarity (symmetric)', 'Containment — row % found in column'),
        horizontal_spacing=0.12
    )
    for mat, col, cscale, cb_x in [
        (jaccard_mat, 1, 'Blues',   0.44),
        (contain_mat, 2, 'Oranges', 1.01),
    ]:
        fig.add_trace(go.Heatmap(
            z=mat, x=labels, y=labels,
            colorscale=cscale, zmin=0, zmax=1,
            text=np.round(mat, 2),
            texttemplate='%{text}',
            hovertemplate='%{y} vs %{x}<br>Value: %{z:.3f}<extra></extra>',
            showscale=True, colorbar=dict(x=cb_x, len=0.9)
        ), row=1, col=col)

    fig.update_layout(
        title=f'Variant overlap — {label} runs',
        height=max(500, 28 * n),
        width=max(900, 40 * n),
    )
    fig.update_xaxes(tickangle=45)
    fig.show()

    jaccard_df = pd.DataFrame(jaccard_mat, index=labels, columns=labels)
    upper = (jaccard_df.where(np.triu(np.ones(jaccard_df.shape), k=1).astype(bool))
             .stack().sort_values(ascending=False))
    print(f'Top 10 most similar {label} run pairs (Jaccard):')
    display(upper.head(10).reset_index().rename(columns={0: 'jaccard', 'level_0': 'Run A', 'level_1': 'Run B'}))

    return run_ids, labels, jaccard_mat, contain_mat, runs_df

# sc/bulk heatmap
sc_run_ids, sc_labels, sc_jaccard_mat, sc_contain_mat, _sc = build_overlap_heatmap(sc_runs, 'sc/bulk')

# WES heatmap
wes_run_ids, wes_labels, wes_jaccard_mat, wes_contain_mat, _wes = build_overlap_heatmap(wes_runs, 'WES')

## Similarity distributions: same-donor vs different-donor
Several cell lines in this dataset share a donor of origin or a parental line.
Same-donor pairs should show higher variant overlap than unrelated pairs.

| Group | Cell lines |
|---|---|
| Donor A | SW480, SW620 |
| Donor B | IS1, IS2, IS3 |
| Donor C | COLO201, COLO205 |
| Parental line | HCT8, HCT15, DLD1 |

In [ ]:
from scipy.stats import gaussian_kde as _gkde

SAME_DONOR_GROUPS = [
    ['SW480',  'SW620'],
    ['IS1',    'IS2',   'IS3'],
    ['COLO201','COLO205'],
    ['HCT8',   'HCT15', 'DLD1'],
]

x_grid = np.linspace(0, 1, 300)

def _add_dist_panel(fig, vals, row, col, color, name=None, show_legend=False, x_title='Similarity'):
    if len(vals) == 0:
        return
    kde_y = _gkde(vals)(x_grid) if len(vals) > 1 else np.zeros_like(x_grid)
    med   = float(np.median(vals))
    ax_num = '' if col == 1 else str(col)
    fig.add_trace(go.Histogram(
        x=vals, histnorm='probability density', nbinsx=30,
        marker_color=color, opacity=0.25, showlegend=False
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=x_grid, y=kde_y, mode='lines', name=name,
        line=dict(color=color, width=2.5),
        legendgroup=name, showlegend=show_legend
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=[med, med], y=[0, kde_y.max() * 1.15 if kde_y.max() > 0 else 1],
        mode='lines', line=dict(color=color, width=1.5, dash='dot'),
        legendgroup=name, showlegend=False, hoverinfo='skip'
    ), row=row, col=col)
    fig.add_annotation(
        x=med, y=kde_y.max() * 1.18 if kde_y.max() > 0 else 1.1,
        text=f'median {med:.3f}  n={len(vals)}',
        showarrow=False, font=dict(size=10, color=color),
        xref=f'x{ax_num}', yref=f'y{ax_num}'
    )
    fig.update_xaxes(title_text=x_title, row=row, col=col)

if sc_run_ids is None:
    print('Run the heatmap cell first.')
else:
    n = len(sc_run_ids)

    # ── Build same/diff sets for sc/bulk ─────────────────────────────────────
    same_donor_pairs = set()
    for grp in SAME_DONOR_GROUPS:
        present = [lb for lb in grp if lb in sc_labels]
        for i in range(len(present)):
            for j in range(i + 1, len(present)):
                same_donor_pairs.add(frozenset([present[i], present[j]]))

    print(f'Same-donor pairs present in sc/bulk data: {len(same_donor_pairs)}')
    for p in sorted(same_donor_pairs, key=lambda x: sorted(x)[0]):
        print(f'  {sorted(p)}')

    jac_same, jac_diff = [], []
    for i in range(n):
        for j in range(i + 1, n):
            pair = frozenset([sc_labels[i], sc_labels[j]])
            (jac_same if pair in same_donor_pairs else jac_diff).append(sc_jaccard_mat[i, j])

    asym_same, asym_diff = [], []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            pair = frozenset([sc_labels[i], sc_labels[j]])
            (asym_same if pair in same_donor_pairs else asym_diff).append(sc_contain_mat[i, j])

    jac_same  = np.array(jac_same);  jac_diff  = np.array(jac_diff)
    asym_same = np.array(asym_same); asym_diff = np.array(asym_diff)

    print(f'\nJaccard:     {len(jac_same)} same-donor pairs, {len(jac_diff)} other pairs')
    print(f'Containment: {len(asym_same)} same-donor values, {len(asym_diff)} other values')

    # Mann-Whitney
    stat_j, pval_j = stats.mannwhitneyu(jac_same, jac_diff, alternative='greater')
    stat_a, pval_a = stats.mannwhitneyu(asym_same, asym_diff, alternative='greater')
    print(f'\nJaccard   Mann-Whitney p = {pval_j:.4e}')
    print(f'Containment Mann-Whitney p = {pval_a:.4e}')

    SAME_COLOR = '#e76f51'
    DIFF_COLOR = '#457b9d'
    WES_COLOR  = '#6d597a'

    # ── Plot 1: Same-donor pairs only ────────────────────────────────────────
    fig_same = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Jaccard — same-donor pairs  ({len(jac_same)} pairs)',
            f'Containment — same-donor pairs  ({len(asym_same)} values)'
        ),
        horizontal_spacing=0.14
    )
    _add_dist_panel(fig_same, jac_same,  1, 1, SAME_COLOR, x_title='Similarity')
    _add_dist_panel(fig_same, asym_same, 1, 2, SAME_COLOR, x_title='Containment')
    fig_same.update_xaxes(range=[0, 1])
    fig_same.update_yaxes(title_text='Density', row=1, col=1)
    fig_same.update_layout(
        height=360,
        title='Same-donor cell line pairs (SW480/SW620 · IS1/IS2/IS3 · COLO201/COLO205 · HCT8/HCT15/DLD1)'
    )
    fig_same.show()

    # ── Plot 2: Different-donor pairs only ───────────────────────────────────
    fig_diff = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Jaccard — different-donor pairs  ({len(jac_diff)} pairs)',
            f'Containment — different-donor pairs  ({len(asym_diff)} values)'
        ),
        horizontal_spacing=0.14
    )
    _add_dist_panel(fig_diff, jac_diff,  1, 1, DIFF_COLOR, x_title='Similarity')
    _add_dist_panel(fig_diff, asym_diff, 1, 2, DIFF_COLOR, x_title='Containment')
    fig_diff.update_xaxes(range=[0, 1])
    fig_diff.update_yaxes(title_text='Density', row=1, col=1)
    fig_diff.update_layout(
        height=360,
        title='Different-donor pairs (same-donor intersections excluded)'
    )
    fig_diff.show()

    # ── Plot 3: Overlay — same vs diff vs WES ────────────────────────────────
    # WES: all upper-triangle pairs (no same-donor grouping)
    if wes_run_ids is not None and len(wes_run_ids) >= 2:
        m = len(wes_run_ids)
        triu_wes = np.triu_indices(m, k=1)
        wes_jac_vals  = wes_jaccard_mat[triu_wes]
        wes_asym_vals = wes_contain_mat[~np.eye(m, dtype=bool)]
    else:
        wes_jac_vals = wes_asym_vals = np.array([])

    fig_overlay = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Jaccard similarity', 'Containment'),
        horizontal_spacing=0.14
    )

    for col_idx, (same_v, diff_v, wes_v, x_title) in enumerate([
        (jac_same,  jac_diff,  wes_jac_vals,  'Similarity'),
        (asym_same, asym_diff, wes_asym_vals, 'Containment'),
    ], start=1):
        ax_num = '' if col_idx == 1 else str(col_idx)
        for vals, color, name in [
            (same_v, SAME_COLOR, 'Same donor (sc/bulk)'),
            (diff_v, DIFF_COLOR, 'Different donor (sc/bulk)'),
            (wes_v,  WES_COLOR,  'WES (all pairs)'),
        ]:
            if len(vals) == 0:
                continue
            kde_y = _gkde(vals)(x_grid) if len(vals) > 1 else np.zeros_like(x_grid)
            med   = float(np.median(vals))
            fig_overlay.add_trace(go.Histogram(
                x=vals, histnorm='probability density', nbinsx=30,
                marker_color=color, opacity=0.15, showlegend=False
            ), row=1, col=col_idx)
            fig_overlay.add_trace(go.Scatter(
                x=x_grid, y=kde_y, mode='lines', name=name,
                line=dict(color=color, width=2.5),
                legendgroup=name, showlegend=(col_idx == 1)
            ), row=1, col=col_idx)
            fig_overlay.add_trace(go.Scatter(
                x=[med, med], y=[0, kde_y.max() * 1.15 if kde_y.max() > 0 else 1],
                mode='lines', line=dict(color=color, width=1.5, dash='dot'),
                legendgroup=name, showlegend=False, hoverinfo='skip'
            ), row=1, col=col_idx)
            fig_overlay.add_annotation(
                x=med, y=kde_y.max() * 1.18 if kde_y.max() > 0 else 1.1,
                text=f'median {med:.3f}',
                showarrow=False, font=dict(size=10, color=color),
                xref=f'x{ax_num}', yref=f'y{ax_num}'
            )
        fig_overlay.update_xaxes(range=[0, 1], title_text=x_title, row=1, col=col_idx)

    fig_overlay.update_yaxes(title_text='Density', row=1, col=1)
    fig_overlay.update_layout(
        height=400, barmode='overlay',
        title='Same-donor vs different-donor — overlay comparison',
        legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='center', x=0.5)
    )
    fig_overlay.show()

    # ── Summary table ─────────────────────────────────────────────────────────
    rows_s = []
    for gname, jv, av in [
        ('Same-donor (sc/bulk)',      jac_same, asym_same),
        ('Different-donor (sc/bulk)', jac_diff, asym_diff),
    ]:
        rows_s.append({
            'Group': gname,
            'n_jaccard_pairs': len(jv),
            'median_jaccard':  round(float(np.median(jv)), 3) if len(jv)  else None,
            'mean_jaccard':    round(float(np.mean(jv)),   3) if len(jv)  else None,
            'n_containment':   len(av),
            'median_contain':  round(float(np.median(av)), 3) if len(av)  else None,
            'mean_contain':    round(float(np.mean(av)),   3) if len(av)  else None,
        })
    if len(wes_jac_vals) > 0:
        rows_s.append({
            'Group': 'WES (all pairs)',
            'n_jaccard_pairs': len(wes_jac_vals),
            'median_jaccard':  round(float(np.median(wes_jac_vals)),  3),
            'mean_jaccard':    round(float(np.mean(wes_jac_vals)),    3),
            'n_containment':   len(wes_asym_vals),
            'median_contain':  round(float(np.median(wes_asym_vals)), 3),
            'mean_contain':    round(float(np.mean(wes_asym_vals)),   3),
        })
    display(pd.DataFrame(rows_s).set_index('Group'))

## Raw SQL explorer

In [ ]:
sql_box = widgets.Textarea(
    value="SELECT v.chromosome, COUNT(*) AS n\nFROM genotype_calls g\nJOIN variants v ON g.variant_id = v.variant_id\nWHERE g.run_id = (SELECT run_id FROM runs LIMIT 1) AND g.filter_status='PASS'\nGROUP BY v.chromosome\nORDER BY n DESC;",
    layout=widgets.Layout(width='100%', height='100px')
)
run_btn = widgets.Button(description='Run query', button_style='primary')

def on_run_query(btn):
    clear_output(wait=False)
    display(sql_box, run_btn)
    try:
        display(query(sql_box.value))
    except Exception as e:
        print(f'Error: {e}')

run_btn.on_click(on_run_query)
display(sql_box, run_btn)